# Workshop — Delta Lake

**Scenario:** You are managing the customer table at RetailHub. Work through every core Delta Lake feature in order: create → enforce schema → evolve schema → add constraints → CRUD → history → time travel → disaster recovery.

## Tasks at a Glance

| # | Topic | Key Command |
|---|-------|-------------|
| 01 | Create Delta table | `write.format("delta").saveAsTable` |
| 02 | Schema Enforcement | `try/except` — Delta rejects incompatible writes |
| 03 | Schema Evolution | `ALTER TABLE ADD COLUMN` |
| 04 | Constraints | `ALTER TABLE ADD CONSTRAINT ... CHECK (...)` |
| 05 | INSERT | `INSERT INTO ... VALUES` |
| 06 | UPDATE | `UPDATE ... SET ... WHERE` |
| 07 | DELETE | `DELETE FROM ... WHERE ... IN` |
| 08 | MERGE INTO | `MERGE INTO ... USING ... ON ... WHEN MATCHED ...` |
| 09 | Version History | `DESCRIBE HISTORY` |
| 10 | Time Travel — version | `SELECT * FROM table VERSION AS OF 0` |
| 11 | Time Travel — timestamp | `SELECT * FROM table TIMESTAMP AS OF 'ts'` |
| 12 | Disaster Recovery | `RESTORE TABLE ... TO VERSION AS OF N` |

## Learning Objectives

By the end of this workshop you will be able to:

- ✅ Create a managed **Delta table** from a DataFrame
- ✅ Demonstrate **Schema Enforcement** and **Schema Evolution**
- ✅ Add **Delta Constraints** (`CHECK`) to enforce business rules
- ✅ Perform full **CRUD operations**: INSERT, UPDATE, DELETE, MERGE INTO
- ✅ Query **version history** with `DESCRIBE HISTORY`
- ✅ **Time Travel** using version number and timestamp
- ✅ **Restore** a table to an earlier version with `RESTORE TABLE`

## Rules

- Replace `# YOUR CODE HERE` with your solution
- Run the **CHECK** cell after each task — all assertions must pass before moving on
- Working table: `lab_delta_cust_ws` (isolated per workshop run — no collision with demo tables)


In [0]:
%run ../setup/00_setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, TimestampType, LongType
)
from datetime import datetime, timedelta

TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.lab_delta_cust_ws"

# Drop any leftover from a previous run
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

print(f"Working table : {TABLE}")
print(f"Dataset path  : {DATASET_PATH}")


---
> **📍 Section 1 / 3 — Delta Table Lifecycle · Tasks 01–04** &nbsp;|&nbsp; `░░░░░░░░░░░░` 0 / 12 done
>
> After this section you will understand: how to create a Delta table, what Schema Enforcement means, when to use Schema Evolution, and how to declare business-rule Constraints.

---

---
## Task 01 — Create the Delta table

Load `customers.csv` from `{DATASET_PATH}/customers/` into a DataFrame using `inferSchema=true`,
then write it as a **managed Delta table** at `TABLE`.

**Goal:** Store `customers_df` and a row count of the persisted table.

| What | How |
|------|-----|
| Read CSV | `spark.read.option("header","true").option("inferSchema","true").csv(path)` |
| Write Delta | `.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(TABLE)` |
| Count | `spark.table(TABLE).count()` |

📖 [Delta Lake quickstart](https://docs.databricks.com/delta/quick-start.html) · [CREATE TABLE USING DELTA](https://docs.databricks.com/sql/language-manual/sql-ref-syntax-ddl-create-table-using.html)


**💡 Guidance — Task 01**

You need to do two things: **read a CSV file into a DataFrame**, then **persist it as a managed Delta table**.

**Reading the file**
`spark.read` is a `DataFrameReader` — think of it as a builder where you chain configuration before specifying the format and path. Two options are essential here: one that tells Spark the first row contains column names, and one that lets Spark automatically infer each column's data type (integers, doubles, strings, dates, etc.) by scanning the data. The final call should specify the CSV format and the full file path using `DATASET_PATH`.

**Writing as a Delta table**
Once you have the DataFrame, use `spark.write` (also a builder pattern). You need to declare three things: the storage format (`delta`), the write behaviour if the table already exists (`overwrite`), and whether to allow the schema to be replaced on overwrite. The last method in the chain should register the table **by name** in Unity Catalog using the `TABLE` variable — this makes it a *managed* table, meaning Databricks owns both the metadata and the data files.

**Things to think about**
- What is the difference between writing to a *path* vs. registering by *name*?
- After writing, can you confirm the row count in the persisted table matches the source DataFrame? How would you read the table back?


In [0]:
# Step 1 — read customers.csv into customers_df
# (hint: use header=true and inferSchema=true options; path = f"{DATASET_PATH}/customers/customers.csv")
customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATASET_PATH}/customers/customers.csv")
)

# Step 2 — persist customers_df as a managed Delta table at TABLE
# (hint: customers_df.write.format("delta"), overwrite mode, allow schema overwrite)
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"Source rows : {customers_df.count():,}")
print(f"Table rows  : {spark.table(TABLE).count():,}")
display(spark.table(TABLE).limit(5))

In [0]:
# ── CHECK 01 ──────────────────────────────────────────────────────────
assert 'customers_df' in dir(), "customers_df not defined"
_tbl = spark.table(TABLE)
assert _tbl.count() > 0, "Delta table is empty"
assert _tbl.count() == customers_df.count(), \
    f"Row mismatch: source={customers_df.count()}, table={_tbl.count()}"
assert 'customer_id' in _tbl.columns, "Missing column: customer_id"
# .collect() is OK here: single-row result from DESCRIBE DETAIL
_fmt = spark.sql(f"DESCRIBE DETAIL {TABLE}").collect()[0]['format']
assert _fmt == 'delta', f"Table format must be 'delta', got '{_fmt}'"
print(f"✓ Task 01 passed — {_tbl.count():,} rows, format: delta")


---
## Task 02 — Schema Enforcement

Delta Lake **rejects writes that violate the current schema** (wrong column names or incompatible types).
Try to append a DataFrame that has an **extra column** `mood STRING` that does not exist in `TABLE`.

**Goal:** Catch the `AnalysisException` raised by Delta and store the error message in `enforcement_error` (str).

| What | How |
|------|-----|
| Build bad DataFrame | `spark.createDataFrame(...)` with extra / mismatched column |
| Trigger rejection | `.write.format("delta").mode("append").saveAsTable(TABLE)` |
| Catch error | `try / except Exception as e:` → `enforcement_error = str(e)` |

📖 [Delta schema validation](https://docs.databricks.com/delta/schema-validation.html)


**💡 Guidance — Task 02**

The goal is to **prove that Delta Lake rejects an incompatible write** and to capture the resulting error.

**Building the "bad" DataFrame**
Create a small DataFrame using `spark.createDataFrame`. Give it at least one column whose name does **not** exist in `TABLE` (e.g. a column called `mood` that was never part of the original schema). You only need one row — the content doesn't matter, only the schema mismatch.

**Triggering and catching the rejection**
Try to append this DataFrame to the table using the standard `.write` chain in `append` mode. Wrap that call inside a `try / except` block. When Delta detects the schema mismatch, it raises an exception — catch it and store its string representation in `enforcement_error`.

**What to observe**
Before the attempt, record the current row count in `rows_before_bad`. After the (failed) write, query the table again and compare. You should see the count is **unchanged** — Delta does not perform partial writes. The entire operation is either committed or rejected atomically.

**Things to think about**
- Why is atomic rejection important for data reliability?
- What would happen if you used `.option("mergeSchema", "true")` instead — would the write still fail?


In [0]:
rows_before_bad = spark.table(TABLE).count()
enforcement_error = None

# Step 1 — create bad_df: a DataFrame that includes a column NOT present in TABLE
# (e.g. add a "mood" column — use spark.createDataFrame with an explicit schema list)
bad_df = spark.createDataFrame(
    [("BAD001", "Test", "Unknown", "test@bad.com", "Nowhere", "happy")],
    ["customer_id", "first_name", "last_name", "email", "country", "mood"]
)

# Step 2 — try to append bad_df to TABLE; catch and store the error message
try:
    bad_df.write.format("delta").mode("append").saveAsTable(TABLE)
except Exception as e:
    enforcement_error = str(e)

print(f"Caught error : {'YES' if enforcement_error else 'NO'}")
print(f"Rows before  : {rows_before_bad:,}  |  Rows now: {spark.table(TABLE).count():,}")

In [0]:
# ── CHECK 02 ──────────────────────────────────────────────────────────
assert enforcement_error is not None, \
    "enforcement_error is None — the write should have raised an exception"
assert isinstance(enforcement_error, str) and len(enforcement_error) > 0, \
    "enforcement_error must be a non-empty string"
_rows_now = spark.table(TABLE).count()
assert _rows_now == rows_before_bad, \
    f"Row count changed! Schema enforcement failed — expected {rows_before_bad}, got {_rows_now}"
print(f"✓ Task 02 passed — Delta rejected bad schema, table unchanged ({_rows_now:,} rows)")


---
## Task 03 — Schema Evolution

Add a new column `loyalty_tier` (STRING) to `TABLE` using `ALTER TABLE`.

**Goal:** The column must exist in the schema after the ALTER statement; existing rows get `NULL`.

| What | How |
|------|-----|
| Add column via SQL | `ALTER TABLE {TABLE} ADD COLUMN column_name STRING` |
| Verify in schema | `spark.table(TABLE).schema` → field list |
| Alternative (write-time) | `.option("mergeSchema", "true")` on a write |

📖 [ALTER TABLE — add columns](https://docs.databricks.com/sql/language-manual/sql-ref-syntax-ddl-alter-table.html) · [Delta schema evolution](https://docs.databricks.com/delta/update-schema.html)


**💡 Guidance — Task 03**

The goal is to **add a new column to an existing Delta table without touching any of the data files**.

**How to add a column**
Delta supports standard SQL DDL. Think about the SQL statement that modifies a table's schema by adding a new column — it follows the pattern `ALTER TABLE … ADD COLUMN …`. Pass this statement to `spark.sql(...)`. The column you need to add is called `loyalty_tier` and should store text values.

**What happens under the hood?**
Unlike a traditional database, Delta does **not** rewrite any Parquet files when you add a column. It only appends a new schema entry to the `_delta_log`. This means the operation is nearly instantaneous regardless of how large the table is. All existing rows will carry a `NULL` for the new column until explicitly updated.

**Verifying the change**
After executing the statement, inspect `spark.table(TABLE).columns` or use `DESCRIBE TABLE {TABLE}` — the new column should be visible. You can also do a quick `display(...)` selecting just `customer_id` and `loyalty_tier` to confirm the `null` values.

**Things to think about**
- When would you prefer `ALTER TABLE ADD COLUMN` over using `.option("mergeSchema", "true")` during a write?
- What is the data type of `loyalty_tier` going to be, and which SQL keyword expresses that?


In [0]:
# Add a new STRING column named loyalty_tier to TABLE
# (use a SQL DDL statement via spark.sql)
spark.sql(f"ALTER TABLE {TABLE} ADD COLUMN loyalty_tier STRING")

display(spark.table(TABLE).select("customer_id", "loyalty_tier").limit(5))

In [0]:
# ── CHECK 03 ──────────────────────────────────────────────────────────
_tbl = spark.table(TABLE)
assert 'loyalty_tier' in _tbl.columns, "Column 'loyalty_tier' was not added"
_fdict = {f.name: f.dataType for f in _tbl.schema.fields}
assert isinstance(_fdict['loyalty_tier'], StringType), \
    f"loyalty_tier must be StringType, got {_fdict['loyalty_tier']}"
_null_count = _tbl.filter(F.col("loyalty_tier").isNull()).count()
assert _null_count == _tbl.count(), \
    f"All existing rows should have NULL loyalty_tier — {_null_count}/{_tbl.count()} are NULL"
print(f"✓ Task 03 passed — loyalty_tier (STRING) added, all {_null_count:,} existing rows = NULL")


---
## Task 04 — Constraints

Delta Lake supports two types of constraints:
- **NOT NULL** — enforced at the column level (set during table creation or ALTER)
- **CHECK** — custom boolean expression enforced on every write

**Goal:** Add a CHECK constraint named `valid_country` that rejects rows where `country IS NULL`.
Then try to insert a row with `country = NULL` and catch the error.

| What | How |
|------|-----|
| Add CHECK constraint | `ALTER TABLE {TABLE} ADD CONSTRAINT valid_country CHECK (country IS NOT NULL)` |
| Try violating it | INSERT a row with `country = NULL` inside try/except |
| See constraints | `SHOW TBLPROPERTIES {TABLE}` → look for `delta.constraints.*` |

📖 [Delta table constraints](https://docs.databricks.com/tables/constraints.html)


**💡 Guidance — Task 04**

The goal is to **declare a data quality rule on the table and then prove it is enforced**.

**Adding a CHECK constraint**
Delta Lake allows you to attach named constraints to a table using a DDL statement. The constraint you need to add should be called `valid_country` and it must ensure that the `country` column is never `NULL`. Look up the `ALTER TABLE … ADD CONSTRAINT … CHECK (…)` syntax and think about what boolean expression expresses "country must have a value".

**Proving enforcement**
Once the constraint is in place, you need to demonstrate that it actually fires. Write a SQL `INSERT` statement that deliberately includes `NULL` as the value for `country`. Wrap this inside a `try / except` block and save the caught error message in `constraint_error`. The row must **not** appear in the table after the failed insert.

**Inspecting active constraints**
Constraints are stored as table properties. Run `SHOW TBLPROPERTIES {TABLE}` and look for keys that begin with `delta.constraints` — this is how Delta persists named rules.

**Things to think about**
- Is a CHECK constraint evaluated at read time or at write time?
- What would happen if the table already contained rows with `NULL` in `country` when you try to add the constraint?
- How is a CHECK constraint different from a NOT NULL constraint?


In [0]:
constraint_error = None

# Step 1 — add a CHECK constraint named 'valid_country' so country IS NOT NULL
spark.sql(f"ALTER TABLE {TABLE} ADD CONSTRAINT valid_country CHECK (country IS NOT NULL)")

# Step 2 — inspect the active constraints on the table
# Run SHOW TBLPROPERTIES and filter for keys starting with 'delta.constraints'
# This confirms the constraint was registered before you try to violate it
spark.sql(f"SHOW TBLPROPERTIES {TABLE}") \
    .filter(F.col("key").startswith("delta.constraints")) \
    .show(truncate=False)

# Step 3 — try to INSERT a row that violates the constraint (country = NULL)
# Use customer_id = 'BAD_C'; catch and store the error in constraint_error
try:
    spark.sql(f"INSERT INTO {TABLE} (customer_id, first_name, last_name, email, country) VALUES ('BAD_C', 'Bad', 'Customer', 'bad@test.com', NULL)")
except Exception as e:
    constraint_error = str(e)

print(f"Constraint violated : {'YES' if constraint_error else 'NO (constraint may not be set)'}")

In [0]:
# ── CHECK 04 ──────────────────────────────────────────────────────────
# .collect() is OK here: SHOW TBLPROPERTIES returns a small bounded set of rows
_props = {r['key']: r['value']
          for r in spark.sql(f"SHOW TBLPROPERTIES {TABLE}").collect()}

# Constraint must be registered under the expected name
assert 'delta.constraints.valid_country' in _props, \
    "Constraint 'valid_country' not found — expected key 'delta.constraints.valid_country' in SHOW TBLPROPERTIES"

# Constraint expression must reference the correct column
_expr = _props['delta.constraints.valid_country'].lower()
assert 'country' in _expr and 'null' in _expr, \
    f"Constraint expression looks wrong: '{_props['delta.constraints.valid_country']}' — should check that country IS NOT NULL"

# Violation must have been caught
assert constraint_error is not None, \
    "constraint_error is None — the violating INSERT should have raised an exception"

# BAD_C must not be in the table
_bad = spark.table(TABLE).filter(F.col("customer_id") == "BAD_C").count()
assert _bad == 0, "Row with NULL country was inserted — constraint did not fire"

print(f"✓ Task 04 passed — constraint 'valid_country' active: {_props['delta.constraints.valid_country']}")
print(f"  Violating INSERT correctly rejected.")


---
> **📍 Section 2 / 3 — CRUD Operations · Tasks 05–08** &nbsp;|&nbsp; `████░░░░░░░░` 4 / 12 done
>
> After this section you will be able to: run `INSERT INTO`, `UPDATE`, `DELETE`, and `MERGE INTO` statements on a live Delta table — and see each operation create a new version in the transaction log.

---

---
## Task 05 — INSERT

Insert **3 test customers** into `TABLE`.
Use customer IDs `"TEST001"`, `"TEST002"`, `"TEST003"` with country `"TestLand"`.

**Goal:** Table row count increases by exactly 3.

| What | How |
|------|-----|
| Insert via SQL | `spark.sql(f"INSERT INTO {TABLE} (col1, col2, ...) VALUES (...), (...)")` |
| Verify count | `spark.table(TABLE).count()` |
| See new version | `DESCRIBE HISTORY {TABLE}` — INSERT creates version N+1 |

📖 [INSERT INTO — Delta DML](https://docs.databricks.com/sql/language-manual/sql-ref-syntax-dml-insert-into.html)


**💡 Guidance — Task 05**

The goal is to **insert three new test customers into the table** using a single SQL statement.

**Writing the INSERT**
Use `spark.sql(...)` to run a standard `INSERT INTO … (column list) VALUES (…), (…), (…)` statement. A multi-row `VALUES` clause lets you insert all three rows in one atomic operation. Make sure your column list includes at least: `customer_id`, `first_name`, `last_name`, `email`, and `country`.

**Constraint awareness**
Remember that the `valid_country` CHECK constraint added in Task 04 is still active. If you omit `country` or pass `NULL` for it, the entire insert will be rejected. All three customers should belong to country `"TestLand"`.

**Customer IDs to use**
The CHECK cell expects customers with IDs `TEST001`, `TEST002`, and `TEST003` — use exactly those strings.

**What to observe**
Before running the insert, capture the current row count. After the insert, compare — the difference should be exactly 3. Also notice that `DESCRIBE HISTORY {TABLE}` gains a new entry with operation `WRITE` and mode `Append`.

**Things to think about**
- Why does `INSERT INTO` create a new version in the Delta log rather than modifying an existing one?
- What would happen if you tried to insert a row with a duplicate `customer_id`? (Delta does not enforce uniqueness by default.)


In [0]:
count_before_insert = spark.table(TABLE).count()

# Insert TEST001, TEST002, TEST003 — all with country='TestLand'
# Use a single INSERT INTO ... (cols) VALUES (...), (...), (...) statement
spark.sql(f"""
    INSERT INTO {TABLE} (customer_id, first_name, last_name, email, country)
    VALUES 
        ('TEST001', 'Alice', 'Test', 'alice@test.com', 'TestLand'),
        ('TEST002', 'Bob', 'Test', 'bob@test.com', 'TestLand'),
        ('TEST003', 'Charlie', 'Test', 'charlie@test.com', 'TestLand')
""")

count_after_insert = spark.table(TABLE).count()
print(f"Before: {count_before_insert:,} | After: {count_after_insert:,} | Inserted: {count_after_insert - count_before_insert}")

In [0]:
# ── CHECK 05 ──────────────────────────────────────────────────────────
_after = spark.table(TABLE).count()
assert _after == count_before_insert + 3, \
    f"Expected {count_before_insert + 3} rows, got {_after}"
# .collect() is OK here: limited to customer_id column for exact-match assertion
_ids = {r['customer_id'] for r in spark.table(TABLE).select("customer_id").collect()}
for _tid in ['TEST001', 'TEST002', 'TEST003']:
    assert _tid in _ids, f"Inserted customer not found: {_tid}"
_hist = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(3).collect()]
assert 'WRITE' in _hist, "INSERT must create a new WRITE version in Delta history"
print(f"✓ Task 05 passed — {_after:,} rows total, INSERT logged in Delta history")


---
## Task 06 — UPDATE

Update `TEST001`: set `country = "Poland"` and `loyalty_tier = "Gold"`.

**Goal:** Only `TEST001` is changed; all other rows stay untouched.

| What | How |
|------|-----|
| Update via SQL | `UPDATE {TABLE} SET col = val, ... WHERE customer_id = 'TEST001'` |
| Always use WHERE | UPDATE without WHERE modifies every row! |
| New Delta version | Each UPDATE commits as a new version in the transaction log |

📖 [UPDATE — Delta DML](https://docs.databricks.com/sql/language-manual/delta-update.html)


**💡 Guidance — Task 06**

The goal is to **modify two columns on a single, specific row** without affecting any other data.

**Writing the UPDATE**
Use `spark.sql(...)` with the `UPDATE … SET … WHERE …` syntax. The `SET` clause lets you assign new values to multiple columns in one statement — separate each `column = value` pair with a comma. The `WHERE` clause is critical: target only `customer_id = 'TEST001'`. An `UPDATE` without a `WHERE` would modify every row in the table.

**What to change**
Set `country` to `'Poland'` and `loyalty_tier` to `'Gold'` on `TEST001`.

**What happens under the hood?**
Delta identifies which Parquet file(s) contain the matching row, rewrites those files with the updated values, and marks the old files as removed in the `_delta_log`. The old file content is still physically present on disk — this is what makes time travel possible.

**What to observe**
After the update, filter the table on `customer_id = 'TEST001'` and inspect the result. Also run `DESCRIBE HISTORY {TABLE}` — you should see a new entry with operation `UPDATE`.

**Things to think about**
- If `TEST001` were stored across multiple Parquet files, would Delta rewrite all of them or only the affected ones?
- How does the presence of the old file on disk enable the time-travel queries you will write later?


In [0]:
# Update TEST001: set country='Poland' and loyalty_tier='Gold'
# (use UPDATE ... SET ... WHERE — always target a specific row with WHERE)
spark.sql(f"""
    UPDATE {TABLE}
    SET country = 'Poland', loyalty_tier = 'Gold'
    WHERE customer_id = 'TEST001'
""")

display(spark.table(TABLE).filter(F.col("customer_id") == "TEST001"))

In [0]:
# ── CHECK 06 ──────────────────────────────────────────────────────────
_row = spark.table(TABLE).filter(F.col("customer_id") == "TEST001").collect()
assert len(_row) == 1, "TEST001 must still exist after UPDATE"
assert _row[0]['country']      == 'Poland', f"country should be 'Poland', got '{_row[0]['country']}'"
assert _row[0]['loyalty_tier'] == 'Gold',   f"loyalty_tier should be 'Gold', got '{_row[0]['loyalty_tier']}'"
_hist = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(5).collect()]
assert 'UPDATE' in _hist, "UPDATE must appear in Delta history"
print("✓ Task 06 passed — TEST001 updated to Poland/Gold, new version in history")


---
## Task 07 — DELETE

Delete customers `"TEST002"` and `"TEST003"` in a **single** `DELETE` statement using `IN`.

**Goal:** Both rows removed; `TEST001` remains intact.

| What | How |
|------|-----|
| Delete via SQL | `DELETE FROM {TABLE} WHERE customer_id IN ('TEST002', 'TEST003')` |
| Physical removal | Does NOT happen immediately — data still exists in old Parquet files |
| Cleanup | Physical removal happens later during `VACUUM` |

📖 [DELETE FROM — Delta DML](https://docs.databricks.com/sql/language-manual/delta-delete-from.html)


**💡 Guidance — Task 07**

The goal is to **remove two specific rows** from the table in a single, atomic SQL statement.

**Writing the DELETE**
Use `spark.sql(...)` with the `DELETE FROM … WHERE …` syntax. To match multiple values for the same column in one condition, use the `IN (value1, value2)` operator. The two customers to remove are `TEST002` and `TEST003`. Make sure `TEST001` is **not** included — it must remain untouched.

**What happens under the hood?**
Delta does **not** physically erase the rows from the Parquet files immediately. Instead, it records the deletion in the `_delta_log` by marking the affected files as "removed" and writing new files that exclude the deleted rows. The original files remain on disk until `VACUUM` is run. This is why you can still time-travel back to see those rows.

**What to observe**
Before the delete, record the row count. After the delete, the count should drop by exactly 2. Verify that `TEST002` and `TEST003` are gone, but `TEST001` is still present. Check `DESCRIBE HISTORY {TABLE}` for a `DELETE` entry.

**Things to think about**
- What would happen if you accidentally ran `DELETE FROM {TABLE}` without a `WHERE` clause?
- Why is it useful that deleted data remains physically on disk for a period of time?


In [0]:
count_before_delete = spark.table(TABLE).count()

# Delete TEST002 and TEST003 in a single DELETE statement using IN (...)
spark.sql(f"""
    DELETE FROM {TABLE}
    WHERE customer_id IN ('TEST002', 'TEST003')
""")

count_after_delete = spark.table(TABLE).count()
print(f"Before: {count_before_delete:,} | After: {count_after_delete:,} | Deleted: {count_before_delete - count_after_delete}")

In [0]:
# ── CHECK 07 ──────────────────────────────────────────────────────────
_remaining = spark.table(TABLE).count()
assert _remaining == count_before_delete - 2, \
    f"Expected {count_before_delete - 2} rows, got {_remaining}"
# .collect() is OK here: limited to customer_id column for exact-match assertion
_ids = {r['customer_id'] for r in spark.table(TABLE).select("customer_id").collect()}
assert 'TEST002' not in _ids, "TEST002 was not deleted"
assert 'TEST003' not in _ids, "TEST003 was not deleted"
assert 'TEST001' in _ids,     "TEST001 should still exist (was not in the DELETE)"
_hist = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(5).collect()]
assert 'DELETE' in _hist, "DELETE must appear in Delta history"
print(f"✓ Task 07 passed — {_remaining:,} rows remain, DELETE logged in history")


---
## Task 08 — MERGE INTO (upsert)

An incoming batch `updates_df` contains updates for existing customers **and** new customers.
Use `MERGE INTO` to atomically upsert them into `TABLE`.

**Goal:**
- **WHEN MATCHED** → update `first_name`, `last_name`, `email`, `country`, `loyalty_tier`
- **WHEN NOT MATCHED** → insert the full row

| What | How |
|------|-----|
| Register source | `updates_df.createOrReplaceTempView("updates_view")` — already done below |
| MERGE syntax | `MERGE INTO target USING source ON target.id = source.id WHEN MATCHED ... WHEN NOT MATCHED ...` |
| Result | Net +1 row (2 matched/updated, 1 new inserted) |

📖 [MERGE INTO — Delta DML](https://docs.databricks.com/sql/language-manual/delta-merge-into.html)


**💡 Guidance — Task 08**

The goal is to **upsert a batch of records atomically** — updating rows that already exist and inserting rows that do not.

**The source data**
`updates_df` has already been created and registered as a temporary view called `updates_view`. You should not modify it. It contains three records: two that match existing `customer_id` values (they should be updated) and one brand-new `customer_id` (it should be inserted).

**Writing the MERGE statement**
`MERGE INTO` is the SQL command for an upsert. Its structure is:
1. `MERGE INTO <target table> AS target` — the table you want to modify
2. `USING <source view> AS source` — the incoming data
3. `ON <join condition>` — how to match target rows to source rows (think: which column uniquely identifies a customer?)
4. `WHEN MATCHED THEN UPDATE SET …` — which columns to overwrite when a match is found
5. `WHEN NOT MATCHED THEN INSERT (…) VALUES (…)` — which columns to populate for a brand-new row

**Columns to include**
The matched update should refresh: `first_name`, `last_name`, `email`, `country`, and `loyalty_tier`. The unmatched insert should include all six columns from the source.

**What to observe**
The net effect should be +1 row (2 updated, 1 inserted). After the MERGE, check that `TEST001` now has `first_name = 'MergedAlice'` and `loyalty_tier = 'Platinum'`, and that `MERGE_NEW` appears in the table.

**Things to think about**
- Why is `MERGE INTO` preferred over a `DELETE + INSERT` approach for upserts?
- What would happen if two source rows matched the same target row?


In [0]:
# Incoming batch — do not modify
updates_data = [
    ("TEST001",    "MergedAlice",  "Smith",  "merged@test.com",   "France",  "Platinum"),  # UPDATE
    (customers_df.first()["customer_id"], "MergedProd", "X", "mp@test.com", "Spain", "Silver"),  # UPDATE real row
    ("MERGE_NEW",  "NewViaMerge",  "Lab",    "new@merge.com",     "Germany", None),         # INSERT
]
updates_df = spark.createDataFrame(
    updates_data,
    ["customer_id", "first_name", "last_name", "email", "country", "loyalty_tier"]
)
updates_df.createOrReplaceTempView("updates_view")
display(updates_df)


In [0]:
count_before_merge = spark.table(TABLE).count()

# MERGE updates_view INTO TABLE
# - match on customer_id
# - WHEN MATCHED: update first_name, last_name, email, country, loyalty_tier
# - WHEN NOT MATCHED: insert all 6 columns
spark.sql(f"""
    MERGE INTO {TABLE} AS target
    USING updates_view AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED THEN
        UPDATE SET
            target.first_name = source.first_name,
            target.last_name = source.last_name,
            target.email = source.email,
            target.country = source.country,
            target.loyalty_tier = source.loyalty_tier
    WHEN NOT MATCHED THEN
        INSERT (customer_id, first_name, last_name, email, country, loyalty_tier)
        VALUES (source.customer_id, source.first_name, source.last_name, source.email, source.country, source.loyalty_tier)
""")

count_after_merge = spark.table(TABLE).count()
print(f"Before: {count_before_merge:,} | After: {count_after_merge:,} | Net change: {count_after_merge - count_before_merge}")

In [0]:
# ── CHECK 08 ──────────────────────────────────────────────────────────
_after_merge = spark.table(TABLE).count()
assert _after_merge == count_before_merge + 1, \
    f"MERGE should insert 1 new row. Expected {count_before_merge + 1}, got {_after_merge}"
# .collect() is OK here: filtered to a single customer_id — always 0 or 1 row
_t001 = spark.table(TABLE).filter(F.col("customer_id") == "TEST001").collect()
assert len(_t001) == 1, "TEST001 must still exist after MERGE"
assert _t001[0]['first_name']   == 'MergedAlice', \
    f"TEST001 first_name should be 'MergedAlice', got '{_t001[0]['first_name']}'"
assert _t001[0]['loyalty_tier'] == 'Platinum', \
    f"TEST001 loyalty_tier should be 'Platinum', got '{_t001[0]['loyalty_tier']}'"
_new = spark.table(TABLE).filter(F.col("customer_id") == "MERGE_NEW").collect()
assert len(_new) == 1, "MERGE_NEW must be inserted by MERGE"
_hist = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(3).collect()]
assert 'MERGE' in _hist, "MERGE must appear in Delta history"
print(f"✓ Task 08 passed — MERGE: 2 updated, 1 inserted | {_after_merge:,} total rows")


---
> **📍 Section 3 / 3 — History, Time Travel & Recovery · Tasks 09–12** &nbsp;|&nbsp; `████████░░░░` 8 / 12 done
>
> **Final section!** This is what makes Delta Lake unique — a complete transaction log that lets you query or restore any past version of your data. Great for disaster recovery and auditing.

---

---
## Task 09 — Version History

Query the full transaction log and extract the operation timeline.

**Goal:** Store the full history in `full_history`, then compute:
- `version_count` — int, total number of versions
- `operation_list` — Python list of operation names from newest to oldest

| What | How |
|------|-----|
| Full history | `spark.sql(f"DESCRIBE HISTORY {TABLE}")` |
| Sort | `.orderBy(F.desc("version"))` |
| Extract ops | `[r['operation'] for r in df.collect()]` |

📖 [DESCRIBE HISTORY — Delta transaction log](https://docs.databricks.com/delta/history.html)


**💡 Guidance — Task 09**

The goal is to **read the full transaction history of the table** and extract a summary of all committed operations.

**Querying the history**
Delta Lake keeps a complete audit log in the `_delta_log` directory. You can expose it as a regular Spark DataFrame using a specific SQL command — look for `DESCRIBE HISTORY` in the Delta documentation. Assign the result to `full_history`.

**What the history DataFrame contains**
Each row represents one committed transaction. The most useful columns are:
- `version` — an incrementing integer (0 = the initial create)
- `timestamp` — when the operation was committed
- `operation` — a string like `WRITE`, `UPDATE`, `DELETE`, `MERGE`, `ADD CONSTRAINT`, etc.

**Extracting the required variables**
- `version_count` should be the total number of rows in `full_history` — use the appropriate DataFrame method.
- `operation_list` should be a plain Python list of operation strings, ordered newest-first. Think about how to sort a DataFrame by a column in descending order, then collect the values.

**What to observe**
At this point in the workshop you should have at least 7–8 versions (create → add column → add constraint → insert → update → delete → merge → …). The CHECK cell validates that `UPDATE`, `DELETE`, and `MERGE` all appear in `operation_list`.

**Things to think about**
- Which operation name appears for the initial `saveAsTable()` call?
- Does `ALTER TABLE ADD COLUMN` create a new version in the log?


In [0]:
# Query the complete transaction log for TABLE
full_history = spark.sql(f"DESCRIBE HISTORY {TABLE}")

# Total number of committed versions (int)
version_count = full_history.count()

# List of operation names ordered newest → oldest
operation_list = [row['operation'] for row in full_history.orderBy(F.desc("version")).collect()]

print(f"Total versions            : {version_count}")
print(f"Operations (newest first) : {operation_list}")
display(full_history.select("version", "timestamp", "operation").orderBy(F.desc("version")))

In [0]:
# ── CHECK 09 ──────────────────────────────────────────────────────────
assert 'full_history'   in dir(), "full_history not defined"
assert 'version_count'  in dir(), "version_count not defined"
assert 'operation_list' in dir(), "operation_list not defined"
assert isinstance(version_count, int) and version_count >= 5, \
    f"Expected ≥5 versions (CREATE+ALTER+ALTER+INSERT+UPDATE+DELETE+MERGE...), got {version_count}"
assert isinstance(operation_list, list) and len(operation_list) == version_count, \
    "operation_list length must equal version_count"
_ops = set(operation_list)
for _op in ['UPDATE', 'DELETE', 'MERGE']:
    assert _op in _ops, f"'{_op}' not found in history — run the previous tasks first"
print(f"✓ Task 09 passed — {version_count} versions: {operation_list}")


---
## Task 10 — Time Travel by version number

Read `TABLE` **at version 0** (the initial CSV load).
Store the result in `df_v0`.

**Goal:** Prove that time travel returns the state before any DML changes.

| What | How |
|------|-----|
| SQL approach | `SELECT * FROM {TABLE} VERSION AS OF 0` |
| Python approach | `spark.read.format("delta").option("versionAsOf", 0).table(TABLE)` |
| What to verify | `TEST001` does NOT exist in `df_v0`; row count differs from current |

📖 [Delta time travel — VERSION AS OF](https://docs.databricks.com/delta/history.html#retrieve-information-by-using-time-travel)


**💡 Guidance — Task 10**

The goal is to **read the table as it looked at the very beginning** — before any DML statements were applied.

**How time travel works**
Delta stores every version of the data by keeping old Parquet files on disk. You can query any past version using the `VERSION AS OF <number>` clause. Version `0` is the state immediately after the initial `saveAsTable()` call — before the `ADD COLUMN`, before the `INSERT`, before the `UPDATE`, etc.

**Two ways to time-travel**
- SQL approach: embed `VERSION AS OF 0` directly after the table name inside a `SELECT` statement passed to `spark.sql(...)`.
- Python reader approach: use `spark.read.format("delta").option("versionAsOf", 0).table(TABLE)`.

Either approach gives you the same result — a DataFrame reflecting the table at version 0. Assign it to `df_v0`.

**What to observe and verify**
Think about what was in the table at version 0. The CHECK cell will confirm:
- `df_v0` contains a `customer_id` column
- `TEST001` does **not** exist (it was only inserted in Task 05, which created a later version)
- The row count of `df_v0` differs from the current table

**Things to think about**
- Does `df_v0` have the `loyalty_tier` column that was added in Task 03?
- What happens if you request a version number that does not exist yet?


In [0]:
# Read TABLE at version 0 (state right after the initial saveAsTable, before any DML)
# Store the result DataFrame in df_v0
df_v0 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 0")

print(f"Rows at v0      : {df_v0.count():,}")
print(f"Rows in current : {spark.table(TABLE).count():,}")
print(f"Columns at v0   : {df_v0.columns}")

In [0]:
# ── CHECK 10 ──────────────────────────────────────────────────────────
assert 'df_v0' in dir(), "df_v0 not defined"
assert 'customer_id' in df_v0.columns, "df_v0 must have customer_id column"
# .collect() is OK here: limited to customer_id column for exact-match assertion
_v0_ids = {r['customer_id'] for r in df_v0.select("customer_id").collect()}
assert 'TEST001' not in _v0_ids, \
    "TEST001 should NOT be in version 0 (it was inserted in Task 05)"
_v0_count      = df_v0.count()
_current_count = spark.table(TABLE).count()
assert _v0_count != _current_count, \
    "Version 0 and current should have different row counts (DML changed the table)"
print(f"✓ Task 10 passed — v0: {_v0_count:,} rows | current: {_current_count:,} rows")


---
## Task 11 — Time Travel by timestamp

Using `full_history` from Task 09, find the **timestamp of the INSERT operation** (the version that added TEST001).
Store it in `ts_insert` as a Python `datetime`, then read the table `TIMESTAMP AS OF ts_insert` as `df_ts`.

**Goal:** `df_ts` must contain `TEST001` with `country = "TestLand"` (state right after INSERT, before UPDATE).

| What | How |
|------|-----|
| Find INSERT version | `full_history.filter(F.col("operation") == "WRITE")` — multiple WRITE are possible, pick the one that added TEST rows |
| Get timestamp | `.collect()[0]['timestamp']` → Python `datetime` |
| Read by time | `spark.sql(f"SELECT * FROM {TABLE} TIMESTAMP AS OF '{ts_str}'")`  |

📖 [Delta time travel — TIMESTAMP AS OF](https://docs.databricks.com/delta/history.html#retrieve-information-by-using-time-travel)


**💡 Guidance — Task 11**

The goal is to **find the exact timestamp when `TEST001` was first inserted**, then use that timestamp to query the table at that precise moment in history.

**Finding the right timestamp**
You already have `full_history` from Task 09. Filter it to find the version that corresponds to the `INSERT INTO` from Task 05. Think about which `operation` value and which `operationParameters` field identify an append-style write. There may be multiple `WRITE` versions — you want the one created by an `Append` mode operation (not an `Overwrite`).

Collect the filtered result (it should be one row) and extract the `timestamp` field. Delta stores it as a Python `datetime` object.

**Reading the table by timestamp**
Use `TIMESTAMP AS OF '…'` inside a `spark.sql(...)` query. You need to format the `datetime` as a string first — use `strftime` with the format `"%Y-%m-%d %H:%M:%S"`. To avoid edge cases where the timestamp lands exactly on a version boundary, consider adding one second before formatting.

Assign the result to `df_ts`.

**What to verify**
At the INSERT timestamp, `TEST001` should exist (it was just inserted) but its `country` should still be `'TestLand'` — the UPDATE that changed it to `'Poland'` happened in a **later** version.

**Things to think about**
- Why does adding `timedelta(seconds=1)` to the timestamp help?
- What is stored in `operationParameters` and how does `mode` differ between `INSERT INTO` and `saveAsTable(mode="overwrite")`?


In [0]:
# Step 1 — find the timestamp of the WRITE/Append version from full_history
# (filter on operation == "WRITE" and operationParameters.mode == "Append")
insert_row = full_history.filter(
    (F.col("operation") == "WRITE") & 
    (F.col("operationParameters.mode") == "Append")
).orderBy("version").first()
ts_insert = insert_row['timestamp']

# Step 2 — read TABLE at that timestamp; store the result in df_ts
# (use TIMESTAMP AS OF with a formatted timestamp string)
ts_str = (ts_insert + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S")
df_ts = spark.sql(f"SELECT * FROM {TABLE} TIMESTAMP AS OF '{ts_str}'")

print(f"Timestamp used : {ts_insert}")
print(f"Rows in df_ts  : {df_ts.count():,}")

In [0]:
# ── CHECK 11 ──────────────────────────────────────────────────────────
assert 'ts_insert' in dir(), "ts_insert not defined"
assert 'df_ts'     in dir(), "df_ts not defined"
assert isinstance(ts_insert, datetime), f"ts_insert must be datetime, got {type(ts_insert)}"
# .collect() is OK here: limited to customer_id column; second collect filtered to 1 row
_ts_ids = {r['customer_id'] for r in df_ts.select("customer_id").collect()}
assert 'TEST001' in _ts_ids, "df_ts must contain TEST001 (inserted in Task 05)"
# At INSERT time, TEST001.country should still be 'TestLand' (UPDATE came later)
_row = df_ts.filter(F.col("customer_id") == "TEST001").collect()[0]
assert _row['country'] == 'TestLand', \
    f"At INSERT time, TEST001.country should be 'TestLand' (before UPDATE), got '{_row['country']}'"
print(f"✓ Task 11 passed — timestamp travel OK | TEST001.country='TestLand' at insert time")


---
## Task 12 — RESTORE — Disaster Recovery

Simulate a mistake: someone runs `DELETE FROM {TABLE}` **without a WHERE clause** — all rows gone!

Then use `RESTORE TABLE` to roll back to the version just **before** the accidental delete.

**Goal:** Table is back to its pre-delete state (same row count as before the accident).

| What | How |
|------|-----|
| Simulate disaster | `spark.sql(f"DELETE FROM {TABLE}")` — no WHERE → deletes everything |
| Find safe version | `DESCRIBE HISTORY` → look for the latest version before the delete |
| Restore | `RESTORE TABLE {TABLE} TO VERSION AS OF {safe_version}` |
| Verify | Row count after RESTORE equals pre-delete count |

📖 [RESTORE TABLE — Delta Lake](https://docs.databricks.com/sql/language-manual/delta-restore.html)


**💡 Guidance — Task 12**

The goal is to **simulate an accidental full-table delete and then recover** using Delta's restore capability.

**Step 1 — Record the safe state**
Before triggering the disaster, capture two things: the current row count (already stored in `rows_before_disaster`) and the current latest version number. Query `DESCRIBE HISTORY {TABLE}`, sort by version descending, take the first row, and extract the `version` field. Store it in `safe_version` — this is the version you will restore to.

**Step 2 — Simulate the disaster**
Run a `DELETE FROM {TABLE}` statement **with no `WHERE` clause**. This removes every single row. After running it, query the row count — it should be 0. This simulates a real-world accident (a developer running the wrong script in production).

**Step 3 — Restore**
Delta provides a `RESTORE TABLE` command that rolls the table back to any previous version. Use `RESTORE TABLE {TABLE} TO VERSION AS OF {safe_version}`. This is not a manual rebuild — Delta simply re-points the active snapshot to the old version entry in the transaction log, reusing the already-existing Parquet files.

**What to observe**
After the restore, the row count should match `rows_before_disaster` exactly. Also check `DESCRIBE HISTORY {TABLE}` — the restore itself creates a new version with operation `RESTORE`.

**Things to think about**
- Does `RESTORE` delete the "disaster version" from the history, or is it still visible?
- What is the upper limit on how far back you can restore? (Think about `VACUUM` and the retention policy.)


In [0]:
rows_before_disaster = spark.table(TABLE).count()

# Step 1 — record the current latest version number (this will be safe_version)
safe_version = spark.sql(f"DESCRIBE HISTORY {TABLE}").orderBy(F.desc("version")).first()['version']

# Step 2 — simulate the disaster: DELETE every row (no WHERE clause!)
spark.sql(f"DELETE FROM {TABLE}")
print(f"Rows after disaster : {spark.table(TABLE).count():,}  ← should be 0")

# Step 3 — restore the table to safe_version
spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF {safe_version}")

rows_after_restore = spark.table(TABLE).count()
print(f"Before disaster : {rows_before_disaster:,}")
print(f"After restore   : {rows_after_restore:,}")

In [0]:
# ── CHECK 12 ──────────────────────────────────────────────────────────
_restored = spark.table(TABLE).count()
assert _restored == rows_before_disaster, \
    f"After RESTORE, expected {rows_before_disaster:,} rows but got {_restored:,}"
assert _restored > 0, "Table must not be empty after restore"
# .collect() is OK here: limited to 5 rows from DESCRIBE HISTORY
_hist_ops = [r['operation'] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").limit(5).collect()]
assert 'RESTORE' in _hist_ops, "RESTORE must appear in Delta history"
print(f"✓ Task 12 passed — disaster recovered! {_restored:,} rows back, RESTORE in history")


---
##  Workshop Complete!

**Next:** `05_orchestration_jobs.ipynb` — Workflows, Lakeflow Pipelines
